# 1. 图像、特征提取与影像模型

本教程生成 120 例**合成** NIfTI 图像、ROI 和临床记录。它用于验证软件，不能解释成临床研究结果。

准备环境：在仓库根目录执行 `python -m pip install -e ".[image,test]"`。从 `examples/` 目录启动 Jupyter，按 01 → 04 顺序运行。

真实研究只需要替换图像目录和临床表。样本 ID 必须精确匹配；每行是一位独立患者。本版不处理同一患者多病灶/多扫描。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import ov_radiomics as rad

# Run notebooks in order, with examples/ as the working directory.
output = Path("output/notebook_demo")

## 数据与固定划分
示例生成器使用固定种子，提前按结局分层保留 25% 测试集。生成器拒绝覆盖已有数据；重新运行时请换一个新的 `output` 路径。

In [ ]:
from demo import make_data
image_root, clinical = make_data(output, n=120)
clinical.groupby(["group", "label"]).size()

## 配对与检查
`match_images` 按 `.nii` / `.nii.gz` 文件名匹配，`check_images` 检查尺寸、spacing、origin、direction 及 ROI。默认不会自动修正 mask。

In [ ]:
samples = rad.match_images(image_root)
qc = rad.check_images(samples, roi_label=1)
assert qc.ok.all(), qc.loc[~qc.ok]
qc.head()

## 提取参数
`params` 接受 PyRadiomics YAML 文件或字典。这里仅提取 Original 的少量特征以缩短教程时间；这不是 CT/MR 的通用推荐参数。可自行增加 LoG/Wavelet、纹理类别，并明确设置重采样和 binWidth。

In [ ]:
params = {
    "imageType": {"Original": {}},
    "featureClass": {
        "firstorder": ["Mean", "Median", "Variance", "Skewness", "Kurtosis", "Entropy"],
        "shape": ["VoxelVolume", "Sphericity"],
    },
    "setting": {"binWidth": 5},
}
extracted = rad.extract(samples, params=params, roi_label=1, n_jobs=1)
extracted.save(output / "extraction")
extracted.features.head()

In [ ]:
rad.plot_feature_summary(extracted.features)
plt.show()

## 训练内筛选和交叉验证
依次做缺失值处理、Z-score、单因素检验、绝对相关性过滤、LASSO。

教程使用宽松的 `p_threshold=1.0`，用于稳定演示。研究应预先确定筛选规则；可用 `univariate=None` 或 `lasso=False` 关闭相应步骤。

`alpha="cv"` 在内部交叉验证中重新拟合筛选流程；外部模型交叉验证也会重新拟合整条 Pipeline。不会先对全体数据筛选。

In [ ]:
data = rad.join_data(extracted.features, clinical[["ID", "label", "group"]])
features = extracted.features.columns.drop("ID").tolist()
selector = rad.FeatureSelector(
    missing="median", univariate="mannwhitney", p_threshold=1.0,
    corr_method="pearson", corr_threshold=0.9,
    lasso=True, alpha="cv", cv=3, alphas=[0.001, 0.01, 0.05], random_state=42,
)
rad_result = rad.train_models(
    data, target="label", features=features, selector=selector,
    models=["LR", "RandomForest"], cv=3, random_state=42,
    model_params={"RandomForest": {"n_estimators": 50}},
)
rad_result.save(output / "rad")
rad_result.cv_results

其他模型名：`SVM`、`KNN`、`ExtraTrees`、`MLP`、`XGBoost`、`LightGBM`。后两者需要 boost 可选依赖。模型比较使用所有折，不能挑选表现最好的测试划分。

In [ ]:
fitted_selector = rad_result.models["LR"]["preprocess"]
print(fitted_selector.selected_features_)
rad.plot_lasso_path(fitted_selector)
rad.plot_lasso_cv(fitted_selector)
rad.plot_feature_weights(fitted_selector.coefficients_)
rad.plot_cv_results(rad_result.cv_results)
plt.show()

In [ ]:
rad.evaluate(rad_result, threshold=0.5)

`train` 是表观性能；`test` 是固定留出集性能。这里的合成数据指标只说明程序可运行，不说明任何模型的临床价值。